# DefectLens · run everything in one notebook
**Before starting: Runtime → Change runtime type → T4 GPU.**

Run the cells top to bottom. Every step saves to Google Drive and skips work that's already done, so if Colab disconnects, re-run the setup cell and then continue from where you were.

Assumes the dataset is already on Drive (from `00_setup_and_data.ipynb`).

### Setup (run every session)
Mounts Google Drive (where data, caches and results live), clones or updates the repo, installs requirements.
**Edit `REPO_URL` once** to point at your GitHub repo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/<your-username>/defectlens.git"   # <-- change this
import os
if not os.path.exists('/content/defectlens'):
    !git clone -q {REPO_URL} /content/defectlens
else:
    !git -C /content/defectlens pull -q
%cd /content/defectlens
!pip install -q -r requirements.txt
!mkdir -p /content/drive/MyDrive/defectlens/work

In [ ]:
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch runtime to T4!')

## 1 · PatchCore (GPU)
Fits the memory bank per category and scores calibration + test images. ~15–30 min.

In [ ]:
!python scripts/run_patchcore.py

## 2 · Classical baseline
Gabor + LBP with per-location Gaussians. Runs on CPU even on this GPU runtime. ~10–20 min.

In [ ]:
!python scripts/run_classical.py

## 3 · Evaluate
Thresholds from held-out normal images, then AUROC, AU-PRO, F1, IoU, figures.

In [ ]:
!python tests/test_metrics.py && python scripts/evaluate.py

## 4 · AU-PRO parity check and same-device speed benchmark

In [ ]:
!python scripts/check_aupro_parity.py

In [ ]:
!python scripts/benchmark_speed.py

## 5 · Look at the results

In [ ]:
from IPython.display import Markdown, Image, display
import glob
RES = "/content/drive/MyDrive/defectlens/work/results"
display(Markdown(open(f"{RES}/metrics.md").read()))
print(open(f"{RES}/aupro_parity.txt").read())
for f in sorted(glob.glob(f"{RES}/figures/*.png")):
    print(f); display(Image(f, width=700))

## 6 · Stretch (only if time): robustness study + coreset ablation

In [ ]:
!python scripts/robustness.py

In [ ]:
!python scripts/ablation_coreset.py

## 7 · Deploy the demo to Hugging Face Spaces
Needs a Hugging Face **write** token saved as a Colab secret named `HF_TOKEN` (🔑 icon in the left bar, enable notebook access). Set `SPACE_ID` below.

In [ ]:
!python scripts/export_demo.py

In [ ]:
import os, shutil, getpass
from huggingface_hub import HfApi
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = getpass.getpass("HF write token: ")

SPACE_ID = "<your-hf-username>/defectlens"   # <-- change this
api = HfApi(token=token)
api.create_repo(SPACE_ID, repo_type="space", space_sdk="docker", exist_ok=True)

stage = "/content/space_stage"
shutil.rmtree(stage, ignore_errors=True)
shutil.copytree("defectlens", f"{stage}/defectlens")
shutil.copytree("configs", f"{stage}/configs")
os.makedirs(f"{stage}/app")
shutil.copy("app/streamlit_app.py", f"{stage}/app/streamlit_app.py")
shutil.copytree("/content/drive/MyDrive/defectlens/work/demo_artifacts", f"{stage}/app/artifacts")
shutil.copy("deploy/Dockerfile", f"{stage}/Dockerfile")
shutil.copy("deploy/requirements-space.txt", f"{stage}/requirements-space.txt")
shutil.copy("deploy/SPACE_README.md", f"{stage}/README.md")

api.upload_folder(folder_path=stage, repo_id=SPACE_ID, repo_type="space",
                  commit_message="Deploy DefectLens demo")
print(f"Deployed: https://huggingface.co/spaces/{SPACE_ID}  (first build takes a few minutes)")